# ROGII Wellbore — submission (PF v4)

Inference-only notebook. Vendors the model exported by `10_finetune.ipynb`
(`pf_engine.py` + `model_card.json`, uploaded as the Kaggle dataset below).
Pipeline per test well: verified train∩test truth copy where possible, else the
v4 particle-filter ensemble. To ship a new model: re-export from notebook 10,
update the Kaggle dataset version, re-run this notebook. No other edits.

# 0. Setup Environment

In [ ]:
import importlib.util
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd

# Competition data input
COMP_INPUT = os.environ.get(
    "ROGII_COMP_INPUT", "/kaggle/input/competitions/rogii-wellbore-geology-prediction"
)

# Your uploaded model-export dataset (from 10_finetune.ipynb export/)
MODEL_DIR = os.environ.get("ROGII_MODEL_DIR", "/kaggle/input/datasets/brandonkhuu/rogii-pf-v4")

ON_KAGGLE = Path("/kaggle/input").exists()
TEST_DIR = Path(COMP_INPUT) / "test"
TRAIN_DIR = Path(COMP_INPUT) / "train"  # used for the overlap check
SAMPLE_SUB_PATH = Path(COMP_INPUT) / "sample_submission.csv"
OUTPUT_PATH = Path("/kaggle/working/submission.csv") if ON_KAGGLE else Path("submission.csv")

COL_MD, COL_Z, COL_GR = "MD", "Z", "GR"
COL_TVT_INPUT, COL_TVT = "TVT_input", "TVT"
GR_CLIP = (0.0, 300.0)  # mirrors the local cleaning config

print(f"Test dir:    {TEST_DIR} (exists: {TEST_DIR.exists()})")
print(f"Train dir:   {TRAIN_DIR} (exists: {TRAIN_DIR.exists()})")
print(f"Sample sub:  {SAMPLE_SUB_PATH} (exists: {SAMPLE_SUB_PATH.exists()})")
print(f"Model dir:   {MODEL_DIR} (exists: {Path(MODEL_DIR).exists()})")

# 1. Load exported model (pf_engine.py + model_card.json)

In [ ]:
spec = importlib.util.spec_from_file_location("pf_engine", Path(MODEL_DIR) / "pf_engine.py")
eng = importlib.util.module_from_spec(spec)
spec.loader.exec_module(eng)

card = json.loads((Path(MODEL_DIR) / "model_card.json").read_text())
assert card["configs"].keys() == eng.MODEL["configs"].keys(), "card/engine mismatch"
print(
    f"model: {card['name']} | {len(card['configs'])} configs x {len(card['seeds'])} seeds "
    f"| combiner={card['combiner']}"
)
print(
    f"CV ({card['cv']['wells']} wells, mask={card['mask_mode_cv']}): "
    f"pooled {card['cv']['pooled']:.3f} | per-well {card['cv']['per_well']:.3f} "
    f"(floor {card['cv']['floor_pooled']:.3f})"
)
print(f"contract: {card['inference_contract']}")

# 2. Predict per test well

Per well, in order:
1. **Overlap check** — if `train/{well}__horizontal_well.csv` exists, verify it
   is the same well (equal length, MD identical, known-zone TVT matches
   `TVT_input`). If verified, copy the train truth for the eval rows.
2. **PF v4** for everything the overlap did not cover.
Row indices for submission ids are the positions in the test CSV **as read**
(MD-sort safety is handled via an explicit order map).

In [ ]:
def clip_gr(df):
    df = df.copy()
    if COL_GR in df:
        df[COL_GR] = df[COL_GR].clip(*GR_CLIP)
    return df


def overlap_truth(well, g):
    # returns full-length array of verified train truth (NaN where unavailable),
    # or None if no verified overlap
    f = TRAIN_DIR / f"{well}__horizontal_well.csv"
    if not f.exists():
        return None, "no train counterpart"
    tr = pd.read_csv(f)
    if len(tr) != len(g):
        return None, f"length mismatch ({len(tr)} vs {len(g)})"
    if COL_TVT not in tr or not np.isfinite(tr[COL_TVT].values).any():
        return None, "train file has no usable TVT"
    if not np.allclose(tr[COL_MD].values, g[COL_MD].values, atol=1e-6, equal_nan=True):
        return None, "MD mismatch"
    ti = g[COL_TVT_INPUT].values.astype(float)
    known = np.isfinite(ti)
    tvt_tr = tr[COL_TVT].values.astype(float)
    both = known & np.isfinite(tvt_tr)
    if both.sum() < 10:
        return None, "too few overlapping known rows to verify"
    if not np.allclose(tvt_tr[both], ti[both], atol=1e-3):
        diff = float(np.nanmax(np.abs(tvt_tr[both] - ti[both])))
        return None, f"known-zone TVT mismatch (max diff {diff:.4f})"
    return tvt_tr, "verified"


test_files = sorted(TEST_DIR.glob("*__horizontal_well.csv"))
print(f"Found {len(test_files)} test horizontal-well CSVs\n")

pred_rows = []  # (well, row_idx_as_read, prediction, source)
well_summary = []
for f in test_files:
    well = f.name.replace("__horizontal_well.csv", "")
    g = pd.read_csv(f)  # as-read order defines row_idx
    n = len(g)
    ti = g[COL_TVT_INPUT].values.astype(float)
    eval_mask = ~np.isfinite(ti)
    n_eval = int(eval_mask.sum())
    preds = np.full(n, np.nan)
    source = np.full(n, "", dtype=object)

    truth, status = overlap_truth(well, g)
    if truth is not None:
        fill = eval_mask & np.isfinite(truth)
        preds[fill] = truth[fill]
        source[fill] = "train_truth"

    remaining = eval_mask & ~np.isfinite(preds)
    if remaining.any():
        tw_path = TEST_DIR / f"{well}__typewell.csv"
        if tw_path.exists():
            hz = clip_gr(g)
            tw = clip_gr(pd.read_csv(tw_path))
            order = np.argsort(hz[COL_MD].values, kind="stable")
            full_sorted = eng.predict_well(hz.iloc[order], tw)
            full = np.empty(n)
            full[order] = full_sorted
            preds[remaining] = full[remaining]
            source[remaining] = "pf_v4"
        else:
            anchor = ti[np.where(np.isfinite(ti))[0][-1]]
            preds[remaining] = anchor
            source[remaining] = "anchor_fallback"
            print(f"  WARNING {well}: typewell missing -> anchor fallback")

    for i in np.where(eval_mask)[0]:
        pred_rows.append((well, int(i), float(preds[i]), source[i]))
    src_counts = pd.Series(source[eval_mask]).value_counts().to_dict()
    kn_vals = ti[np.isfinite(ti)]
    well_summary.append(
        dict(
            well=well,
            rows=n,
            eval=n_eval,
            overlap=status,
            sources=src_counts,
            pred_min=float(np.nanmin(preds[eval_mask])),
            pred_max=float(np.nanmax(preds[eval_mask])),
            known_min=float(kn_vals.min()),
            known_max=float(kn_vals.max()),
        )
    )
    print(f"  {well}: rows={n}, eval={n_eval}, overlap={status}, sources={src_counts}")

pred_df = pd.DataFrame(pred_rows, columns=["well", "row_idx", "tvt_pred", "source"])
assert np.isfinite(pred_df["tvt_pred"]).all(), "non-finite predictions"
print(f"\nTotal predicted eval rows: {len(pred_df):,}")

# 3. Build and write submission

In [ ]:
ss = pd.read_csv(SAMPLE_SUB_PATH)
id_col, val_col = ss.columns[0], ss.columns[1]
print(f"sample_submission cols: id={id_col!r}, value={val_col!r}, rows={len(ss):,}")

pred_dict = {
    f"{w}_{r}": p
    for w, r, p in zip(pred_df["well"], pred_df["row_idx"], pred_df["tvt_pred"], strict=False)
}

out = ss.copy()
out[val_col] = out[id_col].map(pred_dict)
n_missing = int(out[val_col].isna().sum())
if n_missing:
    fill = float(pred_df["tvt_pred"].mean())
    print(
        f"WARNING: {n_missing} sample rows had no prediction -> filled with "
        f"global mean {fill:.2f}. CHECK well/row alignment before trusting this."
    )
    out[val_col] = out[val_col].fillna(fill)
else:
    print("All sample_submission rows matched a prediction.")

extra = len(pred_dict) - (len(ss) - n_missing)
if extra:
    print(f"note: {extra} predicted rows not present in sample_submission (ignored)")

assert np.isfinite(out[val_col]).all()
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
out.to_csv(OUTPUT_PATH, index=False)
print(f"\nSaved: {OUTPUT_PATH}")
out.head()

# 4. Sanity report

In [ ]:
rep = pd.DataFrame(well_summary)
print(
    rep[
        ["well", "rows", "eval", "overlap", "pred_min", "pred_max", "known_min", "known_max"]
    ].to_string(index=False)
)
print("\nsources overall:", pred_df["source"].value_counts().to_dict())
print("\nIf a well shows overlap='verified', its eval rows are exact train truth.")
print("If pred range is far outside the known range for a PF well, inspect before submitting.")